# Milestone 1: Data Profiling

**Dataset:** Daily Rainfall Observations from 460 Stations and Flood Event Records across West Java, Indonesia (2020-2025)  
**Sumber:** Mendeley Data DOI: 10.17632/d7j3s64vvr.3 | BMKG, DIBI-BNPB, BPBD Jawa Barat  
**Tujuan:** Melakukan profil awal dataset besar menggunakan Polars LazyFrame dan DuckDB. Mendokumentasikan sumber data di `data/README.md`.

## 1. Setup Path & Import

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

ROOT = Path(r"C:\project_1\data analysis\tugas 1")
DATASET_DIR = ROOT / "data" / "raw"
RAW_XLSX = DATASET_DIR / "01_Data-Ch-Jabar-2020-2025.xlsx"
META_XLSX = DATASET_DIR / "02_West_Java_Rainfall_Station_Metadata_Lengkap.xlsx"

if not RAW_XLSX.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {RAW_XLSX}. "
        "Pastikan file berada di data/raw/."
    )

print(f"Dataset: {RAW_XLSX.name}")
print(f"Ukuran file: {RAW_XLSX.stat().st_size / 1_000_000:.2f} MB")

## 2. Load Data Wide Format

In [ ]:
df_wide = pl.read_excel(RAW_XLSX, sheet_name="DATA")
print(f"Wide shape: {df_wide.shape[0]:,} rows x {df_wide.shape[1]:,} cols")
df_wide.head()

## 3. Identifikasi Kolom Tanggal & Stasiun

In [ ]:
date_col = df_wide.columns[0]
station_cols = [c for c in df_wide.columns if c != date_col]
print(f"Date col: {date_col}")
print(f"Jumlah stasiun: {len(station_cols)}")
print(f"Estimasi baris jika di-unpivot (long format): {len(df_wide) * len(station_cols):,} baris")

## 4. Profil Polars (LazyFrame)

In [ ]:
lazy_data = pl.scan_excel(RAW_XLSX, sheet_name="DATA")
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

In [ ]:
contoh_data = df_wide.head(10)
contoh_data

## 5. Profil DuckDB dengan SUMMARIZE

In [ ]:
con = duckdb.connect()
con.register('df_wide', df_wide.to_pandas())
profil_duckdb = con.execute('SUMMARIZE SELECT * FROM df_wide').df()
con.close()
profil_duckdb.head(15)

## 6. Metadata Stasiun (Quick Check)

In [ ]:
try:
    df_meta = pl.read_excel(META_XLSX, sheet_name="metadata")
    print(f"Metadata shape: {df_meta.shape}")
    df_meta.head()
except Exception as e:
    print(e)

## Catatan Milestone 1

- **Sumber dataset:** Daily Rainfall Observations from 460 Stations and Flood Event Records across West Java, Indonesia (2020-2025), Mendeley Data DOI: 10.17632/d7j3s64vvr.3, lisensi CC BY 4.0.
- **Ukuran & dimensi:** File `01_Data-Ch-Jabar-2020-2025.xlsx` berbentuk wide (Date x 460 stasiun), ~2.192 hari x 460 stasiun.
- **Potensi baris:** Jika di-*unpivot* ke bentuk long (`tanggal`, `station_id`, `rainfall_mm`) akan menghasilkan **~1.008.320 baris** (> 1.000.000 baris) sehingga memenuhi syarat tugas.
- **Cakupan:** Jawa Barat, periode 2020–2025, 460 stasiun BMKG.
- **Pendukung:** Tersedia metadata stasiun, data kejadian banjir, *consensus rainfall-flood*, *missing data summary*, dan *spatial layers* (`.gpkg`).
- **Tools:** Polars dan DuckDB digunakan untuk profiling.
- **Data mentah:** Telah ditempatkan di `data/raw/` sesuai struktur tugas.